# PBEL Course Recommender — Kaggle Training

Builds a **100,000+ row** course recommendation database and trains
`sentence-transformers/all-MiniLM-L6-v2` embeddings.

## Before you run

1. Create a Kaggle Notebook (GPU recommended).
2. **Add Input → Datasets** and attach:
   - **Required:** [`septa97/100k-courseras-course-reviews-dataset`](https://www.kaggle.com/datasets/septa97/100k-courseras-course-reviews-dataset) (~100k+ review rows)
   - **Optional (richer subjects):** [`yosefxx590/coursera-courses-and-skills-dataset-2025`](https://www.kaggle.com/datasets/yosefxx590/coursera-courses-and-skills-dataset-2025)
3. Run all cells.
4. Download from **Output**: `courses.csv`, `vectors.npy`, `metrics.json`
5. Place `courses.csv` + `vectors.npy` in the PBEL project root, then start the server.

The dataset is loaded **only from `/kaggle/input`** — nothing is downloaded in this notebook via curl/API.

## 1. Setup

In [ ]:
!pip install -q sentence-transformers==3.4.1

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUT_DIR = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
MIN_ROWS = 100_000  # require > 1 lakh rows
RANDOM_STATE = 42
BATCH_SIZE = 256  # raise on GPU

assert INPUT_ROOT.exists(), (
    "/kaggle/input missing. Add the Coursera reviews dataset via Add Input."
)

## 2. Load Kaggle inputs (≥ 1 lakh rows)

Looks under `/kaggle/input` for `reviews_by_course.tsv` / `reviews.tsv` from the 100K Coursera Reviews dataset.

In [ ]:
def find_input(*name_parts: str) -> Path | None:
    parts = [p.lower() for p in name_parts]
    hits = [
        p for p in INPUT_ROOT.rglob("*")
        if p.is_file() and all(x in p.name.lower() for x in parts)
    ]
    return hits[0] if hits else None


print("Files under /kaggle/input:")
for p in sorted(INPUT_ROOT.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size / 1e6:.2f} MB)")

reviews_path = find_input("reviews_by_course") or find_input("reviews")
if reviews_path is None:
    raise FileNotFoundError(
        "Could not find reviews_by_course.tsv / reviews.tsv.\n"
        "Add dataset: septa97/100k-courseras-course-reviews-dataset"
    )

print("\nUsing reviews file:", reviews_path)
sep = "\t" if reviews_path.suffix.lower() == ".tsv" else ","
raw = pd.read_csv(reviews_path, sep=sep)
print("Raw shape:", raw.shape)
print("Columns:", list(raw.columns))
raw.head(3)

In [ ]:
# Optional: Coursera Skills 2025 for Subject / Title enrichment
skills_path = None
for cand in INPUT_ROOT.rglob("*.csv"):
    cols_sample = pd.read_csv(cand, nrows=0).columns.str.lower().tolist()
    if "title" in cols_sample and ("subject" in cols_sample or "gained skills" in " ".join(cols_sample)):
        skills_path = cand
        break

skills = None
if skills_path is not None:
    print("Enrichment CSV:", skills_path)
    skills = pd.read_csv(skills_path)
    print("Skills shape:", skills.shape, list(skills.columns))
else:
    print("No skills enrichment CSV found (optional). Continuing with reviews only.")

## 3. Build unified catalog (≥ 100,000 rows)

Each review becomes one searchable course document (course slug + review text).
This yields a DB larger than 1 lakh rows while staying tied to real Coursera courses.

In [ ]:
def slug_to_title(slug: str) -> str:
    s = re.sub(r"[-_]+", " ", str(slug).strip())
    return s.title() if s else "Unknown Course"


def _pick(df, names):
    lower = {c.lower().strip(): c for c in df.columns}
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None


course_col = _pick(raw, ["CourseId", "course_id", "courseId", "Id"])
review_col = _pick(raw, ["Review", "review", "text", "comment"])
label_col = _pick(raw, ["Label", "label", "rating", "Rating"])

if review_col is None:
    raise ValueError(f"No review text column in {list(raw.columns)}")

df = pd.DataFrame()
if course_col:
    df["course_id"] = raw[course_col].astype(str).str.strip()
else:
    df["course_id"] = [f"row{i}" for i in range(len(raw))]

df["course_name"] = df["course_id"].map(slug_to_title)
df["course_about"] = raw[review_col].fillna("").astype(str).str.strip()
df["subject"] = raw[label_col].astype(str) if label_col else "Coursera"

# Drop empty reviews
df = df[df["course_about"].str.len() > 10].copy()

# Enrich names/subjects from skills catalog when slug tokens overlap titles
if skills is not None:
    title_c = _pick(skills, ["Title", "title", "course_name"])
    subj_c = _pick(skills, ["Subject", "subject"])
    skills_c = _pick(skills, ["Gained Skills", "skills", "Skills"])
    if title_c:
        skills = skills.copy()
        skills["_key"] = (
            skills[title_c].astype(str).str.lower()
            .str.replace(r"[^a-z0-9]+", "-", regex=True).str.strip("-")
        )
        meta = skills.drop_duplicates("_key").set_index("_key")
        keys = df["course_id"].str.lower()
        matched = keys.isin(meta.index)
        print(f"Skills title matches: {matched.sum()} / {len(df)}")
        if matched.any():
            df.loc[matched, "course_name"] = keys[matched].map(meta[title_c])
            if subj_c:
                df.loc[matched, "subject"] = keys[matched].map(meta[subj_c]).fillna(df.loc[matched, "subject"])
            if skills_c:
                extra = keys[matched].map(meta[skills_c]).fillna("")
                df.loc[matched, "course_about"] = (
                    df.loc[matched, "course_about"] + " Skills: " + extra.astype(str)
                )

# Popularity from rating label if numeric-ish
if label_col:
    lab = pd.to_numeric(raw.loc[df.index, label_col], errors="coerce")
    df["popularity_score"] = (lab.fillna(lab.median()) / 5.0).clip(0, 1)
else:
    df["popularity_score"] = 0.0
df["enrollment_count"] = 0

df = df.reset_index(drop=True)
# Stable unique ids (one per review row)
df.insert(0, "row_id", [f"c{i:06d}" for i in range(len(df))])
# Keep original course slug as course_id for grouping; use row_id as unique key for vectors
courses = df.rename(columns={"row_id": "course_id", "course_id": "course_slug"})
courses["course_about"] = (
    courses["course_name"] + ". " + courses["course_about"]
).str.replace(r"\s+", " ", regex=True).str.strip()

courses = courses[[
    "course_id", "course_name", "course_about", "subject",
    "popularity_score", "enrollment_count", "course_slug",
]].copy()

print(f"Catalog rows: {len(courses):,}")
print(f"Unique Coursera courses (slug): {courses['course_slug'].nunique():,}")
print(courses["subject"].value_counts().head(10))

if len(courses) < MIN_ROWS:
    raise RuntimeError(
        f"Need ≥ {MIN_ROWS:,} rows, got {len(courses):,}. "
        "Make sure the full 100K Coursera reviews dataset is attached."
    )

courses.head(3)

## 4. Encode with all-MiniLM-L6-v2

GPU accelerator strongly recommended for 100k+ rows.

In [ ]:
model = SentenceTransformer(MODEL_NAME)

texts = (
    courses["course_name"].fillna("") + " " + courses["course_about"].fillna("")
).tolist()

vectors = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

print("vectors:", vectors.shape, "dtype:", vectors.dtype)
print("norm[0] ~ 1.0:", float(np.linalg.norm(vectors[0])))

## 5. Accuracy (same `course_slug` = relevant)

Hold out 5k queries. A hit = neighbor shares the same Coursera course slug.

In [ ]:
EVAL_N = min(5000, max(500, len(courses) // 20))
idx = np.arange(len(courses))
_, test_idx = train_test_split(idx, test_size=EVAL_N, random_state=RANDOM_STATE)
train_mask = np.ones(len(courses), dtype=bool)
train_mask[test_idx] = False
train_idx = np.where(train_mask)[0]

train_vec = vectors[train_idx]
train_slug = courses.iloc[train_idx]["course_slug"].values
test_vec = vectors[test_idx]
test_slug = courses.iloc[test_idx]["course_slug"].values

K_LIST = [5, 10]
knn = NearestNeighbors(n_neighbors=max(K_LIST), metric="cosine", algorithm="brute")
knn.fit(train_vec)
dists, inds = knn.kneighbors(test_vec)

metrics = {
    "n_rows": int(len(courses)),
    "n_unique_courses": int(courses["course_slug"].nunique()),
    "n_test": int(len(test_idx)),
    "model": MODEL_NAME,
    "dataset": "septa97/100k-courseras-course-reviews-dataset",
}

rr = []
hit_sims = []
for k in K_LIST:
    precisions, recalls = [], []
    for i in range(len(test_idx)):
        true = test_slug[i]
        neigh = inds[i, :k]
        hits = train_slug[neigh] == true
        n_hit = int(hits.sum())
        n_rel = int((train_slug == true).sum())
        precisions.append(n_hit / k)
        recalls.append(n_hit / n_rel if n_rel else 0.0)
    metrics[f"precision@{k}"] = round(float(np.mean(precisions)), 4)
    metrics[f"recall@{k}"] = round(float(np.mean(recalls)), 4)

for i in range(len(test_idx)):
    hits = train_slug[inds[i]] == test_slug[i]
    if hits.any():
        rank = int(np.argmax(hits)) + 1
        rr.append(1.0 / rank)
        hit_sims.append(float(1.0 - dists[i, rank - 1]))
    else:
        rr.append(0.0)

metrics["mrr"] = round(float(np.mean(rr)), 4)
metrics["mean_hit_similarity"] = round(float(np.mean(hit_sims)) if hit_sims else 0.0, 4)
print(json.dumps(metrics, indent=2))

## 6. Qualitative check

In [ ]:
def recommend(query: str, k: int = 5):
    q = model.encode([query], normalize_embeddings=True)
    knn_all = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute")
    knn_all.fit(vectors)
    d, ix = knn_all.kneighbors(q)
    rows = []
    for dist, i in zip(d[0], ix[0]):
        r = courses.iloc[i]
        rows.append({
            "similarity": round(float(1 - dist), 4),
            "course_name": r["course_name"],
            "slug": r["course_slug"],
            "snippet": r["course_about"][:120],
        })
    return pd.DataFrame(rows)


display(recommend("machine learning python neural networks"))
display(recommend("financial accounting basics for beginners"))

## 7. Export artifacts for PBEL

Download these three files from the Kaggle **Output** panel and put `courses.csv` + `vectors.npy` in the PBEL repo root.

In [ ]:
# Runtime schema (drop slug helper or keep it — data_store ignores unknown cols on bootstrap if we only write required)
export_cols = [
    "course_id", "course_name", "course_about", "subject",
    "popularity_score", "enrollment_count",
]
courses[export_cols].to_csv(OUT_DIR / "courses.csv", index=False)
np.save(OUT_DIR / "vectors.npy", vectors)
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

for p in [OUT_DIR / "courses.csv", OUT_DIR / "vectors.npy", OUT_DIR / "metrics.json"]:
    print(f"{p.name:15} {p.stat().st_size / 1e6:8.2f} MB")

print(f"\nRows exported: {len(courses):,}  (must be ≥ {MIN_ROWS:,})")
print("Done. Download courses.csv + vectors.npy → PBEL project root.")